In [1]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

'/opt/spark'

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
# precode.py

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .master("local") \
                    .appName("Learning DataFrames") \
                    .getOrCreate()
# данные первого датафрейма 
book = [('Harry Potter and the Goblet of Fire', 'J. K. Rowling', 322),
        ('Nineteen Eighty-Four', 'George Orwell', 382),
        ('Jane Eyre', 'Charlotte Brontë', 159),
        ('Catch-22', 'Joseph Heller',  174),
        ('The Catcher in the Rye', 'J. D. Salinger',  168),
        ('The Wind in the Willows', 'Kenneth Grahame',  259),
        ('The Mayor of Casterbridge', 'Thomas Hardy',  300),
        ('Bad Girls', 'Jacqueline Wilson',  299)
]
# данные второго датафрейма
library = [
        ( 322, "1"),
        ( 250, "2" ),
        (400, "2"),
        (159, "1"),
        (382, "2"),
        (322, "1")
]
# названия атрибутов
columns = ['title', 'author', 'book_id']
columns_library = ['book_id', 'Library_id']
# создаём датафреймы
df = spark.createDataFrame(data=book, schema=columns)
df_library  = spark.createDataFrame(data=library, schema=columns_library )
# напишите ваш код ниже


/opt/spark/conf/spark-env.sh: line 31: hadoop: command not found


26/08/18 21:58:46 WARN Utils: Your hostname, fv4im62q1f2ij1439k95 resolves to a loopback address: 127.0.1.1; using 10.130.0.37 instead (on interface eth0)
26/08/18 21:58:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/18 21:58:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/18 21:58:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [16]:
df_join = df.join(df_library, on=['book_id'], how='left_anti') \
          .select("title") \
          .distinct() \





In [17]:
df_cache = df_join.cache()

26/08/18 22:05:45 WARN CacheManager: Asked to cache already cached data.


In [18]:
df_cache.show()

+--------------------+
|               title|
+--------------------+
|The Mayor of Cast...|
|            Catch-22|
|The Catcher in th...|
|The Wind in the W...|
|           Bad Girls|
+--------------------+



In [19]:
df_cache.explain()

== Physical Plan ==
InMemoryTableScan [title#0]
   +- InMemoryRelation [title#0], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(6) HashAggregate(keys=[title#0], functions=[])
            +- Exchange hashpartitioning(title#0, 200), ENSURE_REQUIREMENTS, [plan_id=571]
               +- *(5) HashAggregate(keys=[title#0], functions=[])
                  +- *(5) Project [title#0]
                     +- *(5) SortMergeJoin [book_id#2L], [book_id#6L], LeftAnti
                        :- *(2) Sort [book_id#2L ASC NULLS FIRST], false, 0
                        :  +- Exchange hashpartitioning(book_id#2L, 200), ENSURE_REQUIREMENTS, [plan_id=556]
                        :     +- *(1) Project [title#0, book_id#2L]
                        :        +- *(1) Scan ExistingRDD[title#0,author#1,book_id#2L]
                        +- *(4) Sort [book_id#6L ASC NULLS FIRST], false, 0
                           +- Exchange hashpartitioning(book_id#6L, 200), ENSURE_REQUIREMENTS, [plan_id=56

### Задание 2 

In [22]:
from pyspark import SparkContext,SparkConf
sc = SparkContext.getOrCreate(SparkConf())
sc.setCheckpointDir(dirName="/user/s18314377/analytics/test_check")

In [23]:
df_checkpoint = df_join.checkpoint()

In [24]:
df_checkpoint.explain()

== Physical Plan ==
*(1) Scan ExistingRDD[title#0]


